# CNH Histology Microscopy — File Renaming Script

Renames exported `.tif` series files and the `.czi` source file according to the CNH naming convention.

THIS VERSION DETECTS AND ACCEPTS PYRAMID SIZES THAT VARY WITHIN ONE FILE
THIS CHANGE WAS MADE APRIL 29TH 2026 7:30PM

**TIF:**  `[DS][Block]_[Cond]_SL[Slide]_IM[Img]_[Stain]_G[R#]_[Obj]_[Focus]_[Date]_[Initials].tif`
**CZI:**  `[DS][Block]_[Cond]_SL[Slide]_[Stain]_[Focus]_[Date]_[Initials].czi`

#### Resolution Pyramid Groups (G[R#])
Each image is exported as a group of files at progressively lower resolutions.
- `G[R1]` = full resolution (largest file)
- `G[R2]`, `G[R3]`... = downsampled levels
- `G[R#]` = thumbnail (smallest file)

These are needed together for fast pan/zoom in whole-slide image viewers (e.g. QuPath).

Example of how it looks in the data: 

- series0   494 MB  ↘
- series1   124 MB   ↘  pyramid group 1 (IM01)
- series2    32 MB    ↘
- series3     9 MB     ↘
- series4     3 MB      ↙ ← smallest
- series5   591 MB  ← SIZE JUMPED UP = new group starts, pyramid size = 5

## Quick Reference — What Each Cell Does

| Cell | Does | Touches disk? |
|---|---|---|
| 1 | Imports | No |
| 2 | Your metadata inputs | No |
| 3 | Finds files, detects pyramid size | No |
| 4 | Builds and prints rename preview | No |
| 5 | Checks for conflicts | No |
| 6 | Renames files (`CONFIRMED = True`) | ✅ Yes |
| 7 | Undoes rename (`UNDO_CONFIRMED = True`) | ✅ Yes |

## Cell 1 - Import 

In [ ]:
import os
import re

## Cell 2 - User Imputs


**This is the only cell you need to edit between folders.**  


Fill in the metadata for the folder you are about to rename.


| Variable | Description | Example |
|---|---|---|
| `folder_path` | Full path to the folder | `"/Users/CNH/data/03p"` |
| `ds_block` | Embedding round + block letter | `"03p"`, `"02aa"` |
| `condition` | Sample condition code | `"E"`, `"Z"`, `"C"`, `"Cad"`, `"D4"` |
| `slide` | Slide number — digits only, `SL` added automatically | `"47"` → `SL47` |
| `slide_padding` | Minimum digit width for slide number | `2` → `SL07`, `3` → `SL047` |
| `stain` | Stain(s), hyphen-separated | `"BF-DAPI"`, `"Phalloidin-Sytox-DAPI"` |
| `objective` | Objective magnification | `"10x"`, `"20x"`, `"40x"` |
| `focus` | Region imaged | `"WB"`, `"HD"`, `"BR"`, `"ABD"`, `"THX"` |
| `date` | Imaging date | `"20260419"` (YYYYMMDD) |
| `initials` | Your initials | `"CNH"` |
| `pyramid_size_override` | Force a pyramid size — leave `None` for auto-detect | `None`, `4`, `5` |

In [ ]:
# ============================================================
# USER INPUTS — edit these for each folder
# ============================================================

folder_path           = r"/Volumes/cnelya_lab/Lab/Analysis/Charlie_Heacock/Slide Scanner/36"

ds_block              = "01o"       # embedding round + block (e.g. 03p, 02aa)
condition             = "E"         # E, Z, C, Cad, D4, D7, etc.
slide                 = "36"        # slide number digits only — SL added automatically
slide_padding         = 2           # zero-padding width (2 → SL47, 3 → SL047)
stain                 = "GAF-DAPI"   # stains hyphen-separated
objective             = "20x"       # objective magnification
focus                 = "WB"        # WB, HD, BR, ABD, THX
date                  = "20250730"  # YYYYMMDD
initials              = "CNH"       # your initials

# Leave as None to auto-detect pyramid size from file sizes
# Set to an integer if auto-detection gets the wrong result

pyramid_size_override = None

## Cell 3 - File Discovery & Pyramid Detection

This cell does three things:

**1. Finds all `.tif` series files** in the folder  
Only files matching the `series#` naming pattern are included — subfolders and 
unrelated files are ignored.

**2. Finds the `.czi` file**  
Warns you if none is found or if more than one is found.

******CHANGE MADE HERE******
**3. Auto-detects the pyramid size**  

Instead of assuming a fixed pyramid size, the script walks through files in series
order and detects each group boundary by looking for where file size jumps back up.

This means groups of 4 and groups of 5 in the same folder are both handled correctly.
Any group whose size differs from the most common size is flagged with a warning.


In [ ]:
# ============================================================
# FILE DISCOVERY & PYRAMID SIZE DETECTION
# ============================================================

# ---- Helper: extract series number from filename ----
def get_series_num(filename):
    match = re.search(r'series(\d+)', filename, re.IGNORECASE)
    return int(match.group(1)) if match else -1


# ---- Discover files ----
all_files = [f for f in os.listdir(folder_path)
             if os.path.isfile(os.path.join(folder_path, f))]

# Only include TIF files that match the series# pattern
tif_files_raw = [f for f in all_files
                 if f.lower().endswith('.tif')
                 and re.search(r'series\d+', f, re.IGNORECASE)
                 and not f.startswith('._')]  # exclude Mac resource fork files
                                             # (macOS makes these hidden '._' copies on network drives)

czi_files = [f for f in all_files if f.lower().endswith('.czi')]


# ---- Sort TIF files by series number ----
tif_files = sorted(tif_files_raw, key=get_series_num)


# ---- Separate the last two standalone files ----
# These are always the last two in the sequence:
#   second to last = SlideLabel
#   last           = SlideOverview
special_tifs = tif_files[-2:]    # pull them out
tif_files    = tif_files[:-2]    # remaining files used for pyramid detection

print(f"  Slide label file    : {special_tifs[0]}")
print(f"  Slide overview file : {special_tifs[1]}")


# ---- Validate discovery ----
if not tif_files:
    raise FileNotFoundError(
        "No TIF series files found.\n"
        "Check that folder_path is correct and that files contain 'series#' in the name."
    )

print(f"✅  Found {len(tif_files)} TIF series files")

if not czi_files:
    print("⚠️  No CZI file found in this folder")
elif len(czi_files) > 1:
    print(f"⚠️  Multiple CZI files found — only the first will be renamed: {czi_files}")
else:
    print(f"✅  Found CZI file : {czi_files[0]}")

#******CHANGE MADE HERE********
# ---- Get file sizes in MB ----
sizes_mb = [os.path.getsize(os.path.join(folder_path, f)) / (1024 ** 2)
            for f in tif_files]


# ---- Dynamic group detection ----
# Walk through files in series order.
# When size[i] > size[i-1] the previous pyramid ended and a new one starts.

groups        = []
current_group = [tif_files[0]]

for i in range(1, len(tif_files)):
    if sizes_mb[i] > sizes_mb[i - 1]:
        groups.append(current_group)
        current_group = [tif_files[i]]
    else:
        current_group.append(tif_files[i])

groups.append(current_group)   # add the final group


# ---- Analyze group sizes ----
group_sizes       = [len(g) for g in groups]
most_common_size  = max(set(group_sizes), key=group_sizes.count)
irregular_groups  = [(i + 1, len(g), g)
                     for i, g in enumerate(groups)
                     if len(g) != most_common_size]


# ---- Summary ----
print(f"  Groups detected              : {len(groups)}")
print(f"  Expected pyramid size        : {most_common_size} levels (most common)")
print(f"  Irregular groups             : {len(irregular_groups)}")


# ---- Flag irregular groups ----
if irregular_groups:
    print(f"\n  ⚠️  IRREGULAR PYRAMID GROUPS DETECTED:\n")
    for im_num, size, files in irregular_groups:
        print(f"    IM{str(im_num).zfill(2)} — {size} levels instead of {most_common_size}:")
        for f in files:
            mb = os.path.getsize(os.path.join(folder_path, f)) / (1024 ** 2)
            print(f"      {f}  ({mb:.1f} MB)")
    print()
    print("  These will still be named correctly.")
    print(f"  G[R#] will go up to however many levels are present for that group.")
    print(f"  e.g. a group of 4 → G[R1] through G[R4]")
    print()
    print("  If this looks wrong (not an intentional short acquisition), check")
    print("  that no files are missing from your folder before running Cell 6.")
else:
    print(f"\n  ✅  All {len(groups)} groups have {most_common_size} pyramid levels\n")


# ---- Print group summary table ----
print(f"\n  {'IM##':<8} {'Levels':<10} {'First file':<50} {'Full res (MB)':>14}  Status")
print("  " + "-" * 95)

for i, g in enumerate(groups):
    im_num      = i + 1
    full_res_mb = os.path.getsize(os.path.join(folder_path, g[0])) / (1024 ** 2)
    status      = "✅" if len(g) == most_common_size else f"⚠️  {len(g)} levels"
    print(f"  IM{str(im_num).zfill(2):<6} {len(g):<10} {g[0]:<50} {full_res_mb:>14.1f}  {status}")


print(f"\n  Slide label    : {special_tifs[0]}")
print(f"  Slide overview : {special_tifs[1]}")




## Cell 4 - Preview


Builds the complete rename plan and prints every **old → new** name pair
**without touching anything on disk.**

### Naming logic:
    series_number  →  IM##  and  G[R#]

    IM## = (series_number ÷ pyramid_size) + 1   ← which image group
    G[R#] = (series_number % pyramid_size) + 1  ← which resolution level

    Example with pyramid_size = 5:
    series0  →  0 ÷ 5 = 0  →  IM01,  0 % 5 = 0  →  G[R1]
    series4  →  4 ÷ 5 = 0  →  IM01,  4 % 5 = 4  →  G[R5]
    series5  →  5 ÷ 5 = 1  →  IM02,  5 % 5 = 0  →  G[R1]
    series7  →  7 ÷ 5 = 1  →  IM02,  7 % 5 = 2  →  G[R3]


The CZI file gets the same base metadata but no `IM##` or `G[R#]`.

#### ⚠️ Check this output carefully before moving to the conflict check and execute cells.

In [ ]:
# ============================================================
# BUILD RENAME PREVIEW
# ============================================================

slide_str = f"SL{slide.zfill(slide_padding)}"


# ---- CHANGED: now takes im_num and r_level directly instead of series_num ----
def build_tif_name(im_num, r_level):
    """Build the new TIF filename from image group number and resolution level."""
    return (
        f"{ds_block}_{condition}_{slide_str}"
        f"_IM{str(im_num).zfill(2)}"
        f"_{stain}"
        f"_G[R{r_level}]"
        f"_{objective}_{focus}_{date}_{initials}.tif"
    )


# ---- UNCHANGED ----
def build_czi_name():
    """Build the new CZI filename from the global metadata variables."""
    return (
        f"{ds_block}_{condition}_{slide_str}"
        f"_{stain}"
        f"_{focus}_{date}_{initials}.czi"
    )


# ---- UNCHANGED ----
def build_special_name(label):
    """Build filename for standalone slide images (label or overview)."""
    return (
        f"{ds_block}_{condition}_{slide_str}"
        f"_{label}"
        f"_{date}_{initials}.tif"
    )


# ---- CHANGED: now iterates over groups detected in Cell 3 ----
tif_preview  = []
group_ranges = []    # stores (im_num, start_index, end_index, group_size)
current_idx  = 0

for im_idx, group in enumerate(groups):
    im_num    = im_idx + 1
    start_idx = current_idx

    for r_idx, old_name in enumerate(group):
        r_level = r_idx + 1
        tif_preview.append((old_name, build_tif_name(im_num, r_level)))
        current_idx += 1

    group_ranges.append((im_num, start_idx, current_idx, len(group)))


# ---- UNCHANGED ----
czi_preview = []
if czi_files:
    czi_preview.append((czi_files[0], build_czi_name()))


# ---- UNCHANGED ----
special_preview = [
    (special_tifs[0], build_special_name("SlideLabel")),
    (special_tifs[1], build_special_name("SlideOverview")),
]


# ---- CHANGED: preview now uses group_ranges and flags irregular groups ----
print("=" * 90)
print("  RENAME PREVIEW")
print("=" * 90)

for im_num, start, end, group_size in group_ranges:
    flag = ("" if group_size == most_common_size
            else f"  ⚠️  {group_size} levels (expected {most_common_size})")
    print(f"\n  --- Image Group IM{str(im_num).zfill(2)} "
          f"({group_size} pyramid levels){flag} ---")

    for old, new in tif_preview[start:end]:
        print(f"  {old}")
        print(f"  → {new}\n")

if czi_preview:
    print(f"\n  --- CZI file ---")
    old, new = czi_preview[0]
    print(f"  {old}")
    print(f"  → {new}")

print(f"\n  --- Standalone files ---")
for old, new in special_preview:
    print(f"  {old}")
    print(f"  → {new}\n")

all_preview = tif_preview + czi_preview + special_preview
print("=" * 90)
print(f"  Total: {len(tif_preview)} TIF + {len(czi_preview)} CZI "
      f"+ {len(special_preview)} standalone = {len(all_preview)} file(s) to rename")
print("=" * 90)

## Cell 5 - Conflict Check


Checks for two types of problems **before anything is renamed**:

1. **Duplicate output names** — two source files would be assigned the same new name.  
   This would cause one to silently overwrite the other.

2. **Overwrite conflicts** — a proposed new name matches a file already in the folder  
   that is *not* part of the current rename operation.

If any conflicts are found → **do not run Cell 6**.  
Conflicts almost always mean the pyramid size is wrong.  
Go back to Cell 2, set `pyramid_size_override`, and re-run Cells 3–5.

In [ ]:
# ============================================================
# CONFLICT CHECK
# ============================================================

old_names_set = set(old for old, _ in all_preview)
existing      = set(os.listdir(folder_path))
conflicts     = []


# ---- Check 1: duplicate output names within the rename plan ----
seen = {}
for old, new in all_preview:
    if new in seen:
        conflicts.append(
            f"DUPLICATE OUTPUT NAME : '{new}'\n"
            f"             assigned to : '{seen[new]}'\n"
            f"                    and : '{old}'"
        )
    else:
        seen[new] = old


# ---- Check 2: output name matches an existing file not in our rename plan ----
for old, new in all_preview:
    if new in existing and new not in old_names_set:
        conflicts.append(
            f"WOULD OVERWRITE EXISTING FILE : '{new}'\n"
            f"  (this file exists on disk and is not part of the current rename)"
        )


# ---- Report ----
if conflicts:
    print("⚠️  CONFLICTS DETECTED — resolve these before running Cell 6:\n")
    for i, c in enumerate(conflicts, 1):
        print(f"  [{i}] ❌  {c}\n")
    print("─" * 60)
    print("Tip: duplicate names usually mean pyramid_size is wrong.")
    print("     Set pyramid_size_override in Cell 2 and re-run Cells 3–5.")
else:
    print("✅  No conflicts detected")
    print(f"    {len(all_preview)} files are ready to rename")
    print("\n    → Set CONFIRMED = True in Cell 6 and run it when ready")

## Cell 6 - Excute 


This cell performs the actual rename operation on disk.

> ⚠️ **Only run this cell after:**
> - The preview in Cell 4 looks exactly right
> - Cell 5 reports no conflicts
> - You have set `CONFIRMED = True` below

Each file is printed as it is renamed. Failures are caught individually so that
one bad file does not stop the rest from being renamed.

In [ ]:
# ============================================================
# EXECUTE RENAME
# ============================================================

# ✋ Set to True only after checking the preview (Cell 4)
#    and confirming no conflicts (Cell 5)
CONFIRMED = True

if not CONFIRMED:
    print("ℹ️  Rename skipped.")
    print("   Set CONFIRMED = True and re-run this cell when you are ready.")

else:
    renamed = []
    errors  = []

    for old_name, new_name in tif_preview + czi_preview + special_preview:
        old_path = os.path.join(folder_path, old_name)
        new_path = os.path.join(folder_path, new_name)

        try:
            os.rename(old_path, new_path)
            print(f"✅  {old_name}")
            print(f"    → {new_name}\n")
            renamed.append((old_name, new_name))
        except Exception as e:
            print(f"❌  FAILED : {old_name}")
            print(f"    Error  : {e}\n")
            errors.append((old_name, str(e)))

    print("=" * 60)
    print(f"  Done — {len(renamed)} renamed, {len(errors)} error(s)")

    if errors:
        print("\n  Files that failed:")
        for name, err in errors:
            print(f"    {name} : {err}")
    else:
        print("\n  All files renamed successfully ✓")
        print("  Run Cell 7 if you need to undo.")

## Cell 7 - Undo

Reverses every rename from Cell 6 by mapping new names back to original names,
using the same `tif_preview` and `czi_preview` lists built in Cell 4.

> ⚠️ **This only works in the same notebook session.**  
> If you close and reopen the notebook the preview lists will be lost.  
> To undo after reopening: re-run Cell 2 with the same inputs, then Cell 3 and 4  
> to rebuild the mapping, then run this cell.

Set `UNDO_CONFIRMED = True` to execute.

In [ ]:
# ============================================================
# UNDO — revert all renames from Cell 6
# ============================================================

# ✋ Set to True to revert all renames
UNDO_CONFIRMED = False

if not UNDO_CONFIRMED:
    print("ℹ️  Undo skipped.")
    print("   Set UNDO_CONFIRMED = True and re-run this cell to revert.")

else:
    reverted  = []
    not_found = []
    errors    = []

    for old_name, new_name in tif_preview + czi_preview + special_preview:
        new_path = os.path.join(folder_path, new_name)
        old_path = os.path.join(folder_path, old_name)

        if not os.path.exists(new_path):
            not_found.append(new_name)
            print(f"⚠️  Not found (already reverted or never renamed?): {new_name}")
            continue

        try:
            os.rename(new_path, old_path)
            print(f"↩️  {new_name}")
            print(f"    → {old_name}\n")
            reverted.append(new_name)
        except Exception as e:
            print(f"❌  FAILED : {new_name}")
            print(f"    Error  : {e}\n")
            errors.append((new_name, str(e)))

    print("=" * 60)
    print(f"  Done — {len(reverted)} reverted, {len(not_found)} not found, {len(errors)} error(s)")